In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
id2int = {'Parasitized': 0, 'Uninfected': 1}

train_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(128),
    transforms.CenterCrop(128),
    transforms.ColorJitter(brightness=(0.95,1.05), contrast=(0.95,1.05), saturation=(0.95,1.05), hue=0.05),
    transforms.RandomAffine(degrees=5, translate=(0.01,0.1), scale=(0.9,1.1), shear=10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
val_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(128),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


In [ ]:
import os
import cv2
from torch import  randint
from pathlib import Path


class Malaria(Dataset):
    def __inti__(self, files, tfms=None):
        self.files = files
        self.tfms = tfms
        
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        path = self.files[idx]
        cls = os.path.basename(Path(path).parent)
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img, cls
    
    def choose(self):
        return self[randint(len(self.files))]
        

In [9]:
from glob import glob
import numpy as np
from torch.utils.data import random_split, dataloader

all_files = glob('cell_images/*/*.png')
# print(len(all_files))
np.random.shuffle(all_files)

train_size = int(0.8*len(all_files))
test_size = int(0.2*len(all_files))

train_data, test_data = random_split(all_files, [train_size, test_size])
train_loader = DataLoader(Malaria(train_data, train_tfms), batch_size=32, shuffle=True)
val_loader = DataLoader(Malaria(test_data, val_tfms), batch_size=32, shuffle=False)


def conv_block(input_channels, out_channels, pool=True):
    return nn.Sequential(
        nn.Dropout(0.1),
        nn.Conv2d(input_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2)
    )

class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
        self.model = nn.Sequential(
            conv_block(3, 16, pool=True),
            conv_block(16, 32, pool=True),
            conv_block(32, 64, pool=True),
            conv_block(64, 128, pool=True),
            conv_block(128, 256, pool=True),
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, len(id2int))
        )



ValueError: Sum of input lengths does not equal the length of the input dataset!